# CARBANAK process-tree graphs

This version deliberately assumes the known CARBANAK layout rather than probing the machine or guessing field names.

It:

1. loads the six fixed JSONL event members and the explicitly listed alert-label workbooks into pandas DataFrames;
2. computes the columns common to all six event DataFrames;
3. builds one directed process graph from `parent_guid -> process_guid` and `process_guid -> childproc_guid` relations;
4. labels every process node with its common event fields, alert status, and alert explanation; and
5. saves each weakly connected process tree/component as a pickle named from the dataset, node count, and root date.

Edit only the configuration cell below when paths or explicit column names change. `READ_NROWS = None` loads every row; set it to an integer only for a quick local test.

In [1]:
from pathlib import Path
import pickle
import tarfile

import networkx as nx
import pandas as pd
from IPython.display import display

# Paths and fixed archive members
DATASET_NAME = "CARBANAK"
DATA_DIR = Path("../../data/CARBANAK")
EVENT_ARCHIVE = DATA_DIR / "edr-events.tar.gz"
GRAPH_DIR = DATA_DIR / "graphs"

EVENT_MEMBERS = {
    "h1": "edr-events/h1-events.jsonl",
    "h2": "edr-events/h2-events.jsonl",
    "h3": "edr-events/h3-events.jsonl",
    "h4": "edr-events/h4-events.jsonl",
    "dc": "edr-events/dc-events.jsonl",
    "fs": "edr-events/fs-events.jsonl",
}

# Add more explicit workbook paths here when needed. No directory search is performed.
ALERT_LABEL_FILES = [DATA_DIR / "edr_alert_labels.xlsx"]

# None means all rows from all six event files.
READ_NROWS = None

# Known event and alert-label columns
PROCESS_GUID = "process_guid"
PARENT_GUID = "parent_guid"
CHILD_GUID = "childproc_guid"
EVENT_TIME = "device_timestamp"

ALERT_GUID = "guid"
ALERT_CLASS = "label"
ALERT_EXPLANATION = "report"
MALICIOUS_LABELS = {"malicious"}

GRAPH_DIR.mkdir(parents=True, exist_ok=True)

## Load the six event DataFrames and alert-label DataFrames

In [2]:
read_json_kwargs = {"lines": True}
if READ_NROWS is not None:
    read_json_kwargs["nrows"] = READ_NROWS

with tarfile.open(EVENT_ARCHIVE, "r:*") as archive:
    event_dfs = {}
    for name, member_name in EVENT_MEMBERS.items():
        member = archive.extractfile(member_name)
        if member is None:
            raise FileNotFoundError(f"Archive member not found: {member_name}")
        with member:
            event_dfs[name] = pd.read_json(member, **read_json_kwargs)

# Named references are convenient while exploring the notebook.
h1_events = event_dfs["h1"]
h2_events = event_dfs["h2"]
h3_events = event_dfs["h3"]
h4_events = event_dfs["h4"]
dc_events = event_dfs["dc"]
fs_events = event_dfs["fs"]

alert_label_dfs = {
    path.name: pd.read_excel(path)
    for path in ALERT_LABEL_FILES
}
alert_labels = pd.concat(alert_label_dfs.values(), ignore_index=True, sort=False)

loaded = pd.DataFrame(
    [(name, len(df), len(df.columns)) for name, df in event_dfs.items()],
    columns=["event_file", "rows", "columns"],
)
print(f"Loaded {len(event_dfs)} event DataFrames and {len(alert_label_dfs)} alert-label DataFrame(s).")
display(loaded)
display(pd.DataFrame({"alert_rows": [len(alert_labels)], "alert_columns": [len(alert_labels.columns)]}))

Loaded 6 event DataFrames and 1 alert-label DataFrame(s).


,event_file,rows,columns
0,h1,3145787,79
1,h2,1275721,79
2,h3,2484316,79
3,h4,2652493,79
4,dc,211946,77
5,fs,112298,53


,alert_rows,alert_columns
0,13702,14


In [3]:
h1_events.head()

,org_key,device_name,device_external_ip,process_path,parent_path,process_cmdline,parent_cmdline,process_username,type,process_guid,...,scriptload_publisher,scriptload_content_length,scriptload_hash,scriptload_content,fileless_scriptload_cmdline,fileless_scriptload_cmdline_length,fileless_scriptload_hash,netconn_remote_device_id,netconn_remote_device_name,alert_id
0,7DMF69PK,PANGOLIN\EFFICIENTGOLDFINCH,130.126.255.205,c:\windows\explorer.exe,c:\windows\system32\userinit.exe,C:\Windows\Explorer.EXE,C:\Windows\system32\userinit.exe,PANGOLIN\efficientgoldfinch,endpoint.event.moduleload,7DMF69PK-0c587426-00001a44-00000000-1da94c854a...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,7DMF69PK,PANGOLIN\EFFICIENTGOLDFINCH,130.126.255.205,c:\windows\system32\svchost.exe,c:\windows\system32\services.exe,C:\Windows\system32\svchost.exe -k netsvcs -p ...,C:\Windows\system32\services.exe,NT AUTHORITY\SYSTEM,endpoint.event.regmod,7DMF69PK-0c587426-00000728-00000000-1da94c81df...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7DMF69PK,PANGOLIN\EFFICIENTGOLDFINCH,130.126.255.205,SYSTEM,NaN,NaN,NaN,NT AUTHORITY\SYSTEM,endpoint.event.regmod,7DMF69PK-0c587426-00000004-00000000-1da94c81b8...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,7DMF69PK,PANGOLIN\EFFICIENTGOLDFINCH,130.126.255.205,c:\windows\system32\svchost.exe,c:\windows\system32\services.exe,C:\Windows\system32\svchost.exe -k netsvcs -p ...,C:\Windows\system32\services.exe,NT AUTHORITY\SYSTEM,endpoint.event.regmod,7DMF69PK-0c587426-00000728-00000000-1da94c81df...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,7DMF69PK,PANGOLIN\EFFICIENTGOLDFINCH,130.126.255.205,c:\windows\system32\svchost.exe,c:\windows\system32\services.exe,C:\Windows\system32\svchost.exe -k netsvcs -p ...,C:\Windows\system32\services.exe,NT AUTHORITY\SYSTEM,endpoint.event.regmod,7DMF69PK-0c587426-00000728-00000000-1da94c81df...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Learn the columns common to all six event files

In [4]:
common_event_columns = sorted(
    set.intersection(*(set(df.columns) for df in event_dfs.values()))
)

print(f"Common columns: {len(common_event_columns)}")
display(pd.DataFrame({"common_event_column": common_event_columns}))

Common columns: 52


,common_event_column
0,action
1,alert_id
2,backend_timestamp
3,childproc_guid
4,childproc_hash
5,childproc_name
6,childproc_pid
7,childproc_publisher
8,childproc_reputation
9,childproc_username


## Build process observations and parent/child edges

For node attributes, the notebook keeps the first observed process row and separately records the earliest parsed event time. Direct parent relations and explicit `childproc_guid` relations are both retained.

In [5]:
def normalize_guid(series: pd.Series) -> pd.Series:
    return (
        series.astype("string")
        .str.strip()
        .str.strip("{}")
        .str.lower()
    )


def parse_event_time(series: pd.Series) -> pd.Series:
    # Carbon Black timestamps are UTC strings ending in " UTC".
    text = series.astype("string").str.strip().str.replace(r"\s+UTC$", "", regex=True)
    return pd.to_datetime(text, errors="coerce", utc=True)


def summarize_edges(
    parents: pd.Series,
    children: pd.Series,
    times: pd.Series,
    event_file: str,
    relation: str,
) -> pd.DataFrame:
    edges = pd.DataFrame({
        "parent": normalize_guid(parents),
        "child": normalize_guid(children),
        "event_time": times,
    }).dropna(subset=["parent", "child"])

    edges = edges[edges["parent"] != edges["child"]]
    if edges.empty:
        return pd.DataFrame(columns=[
            "parent", "child", "first_seen", "last_seen", "n_events",
            "event_files", "relation_types",
        ])

    summary = (
        edges.groupby(["parent", "child"], sort=False)
        .agg(
            first_seen=("event_time", "min"),
            last_seen=("event_time", "max"),
            n_events=("event_time", "size"),
        )
        .reset_index()
    )
    summary["event_files"] = [(event_file,)] * len(summary)
    summary["relation_types"] = [(relation,)] * len(summary)
    return summary


def merge_tuples(values: pd.Series) -> tuple[str, ...]:
    return tuple(sorted({item for group in values for item in group}))


node_value_columns = [column for column in common_event_columns if column != PROCESS_GUID]
node_parts = []
node_count_parts = []
node_time_parts = []
edge_parts = []

for event_file, df in event_dfs.items():
    event_times = parse_event_time(df[EVENT_TIME])
    process_ids = normalize_guid(df[PROCESS_GUID])

    # One compact row per process per event file, retaining the first observed values.
    first_mask = process_ids.notna() & ~process_ids.duplicated()
    node_part = df.loc[first_mask, node_value_columns].copy()
    node_part.insert(0, "node_id", process_ids.loc[first_mask].astype(str).to_numpy())
    node_part["event_file"] = event_file
    node_parts.append(node_part)

    counts = process_ids.dropna().value_counts().rename("event_count").rename_axis("node_id").reset_index()
    node_count_parts.append(counts)

    times = pd.DataFrame({"node_id": process_ids, "first_seen": event_times}).dropna(subset=["node_id"])
    times = times.groupby("node_id", sort=False, as_index=False)["first_seen"].min()
    node_time_parts.append(times)

    edge_parts.append(summarize_edges(
        df[PARENT_GUID], df[PROCESS_GUID], event_times, event_file, "parent_guid->process_guid"
    ))

    if CHILD_GUID in df.columns:
        edge_parts.append(summarize_edges(
            df[PROCESS_GUID], df[CHILD_GUID], event_times, event_file, "process_guid->childproc_guid"
        ))

# Node attributes from the first event-file observation of each process.
node_rows = (
    pd.concat(node_parts, ignore_index=True, sort=False)
    .drop_duplicates("node_id", keep="first")
    .set_index("node_id")
)

node_counts = (
    pd.concat(node_count_parts, ignore_index=True)
    .groupby("node_id", sort=False)["event_count"]
    .sum()
)

node_event_files = (
    pd.concat([
        part[["node_id", "event_file"]]
        for part in node_parts
    ], ignore_index=True)
    .groupby("node_id", sort=False)["event_file"]
    .agg(lambda values: tuple(sorted(set(values))))
)

node_first_seen = (
    pd.concat(node_time_parts, ignore_index=True)
    .groupby("node_id", sort=False)["first_seen"]
    .min()
)

# Merge repeated edge observations across event files and relation types.
edge_table = pd.concat(edge_parts, ignore_index=True, sort=False)
edge_table = (
    edge_table.groupby(["parent", "child"], sort=False)
    .agg(
        first_seen=("first_seen", "min"),
        last_seen=("last_seen", "max"),
        n_events=("n_events", "sum"),
        event_files=("event_files", merge_tuples),
        relation_types=("relation_types", merge_tuples),
    )
    .reset_index()
)

# Parent-only or child-only nodes still need a first-seen time.
endpoint_times = pd.concat([
    edge_table[["parent", "first_seen"]].rename(columns={"parent": "node_id"}),
    edge_table[["child", "first_seen"]].rename(columns={"child": "node_id"}),
], ignore_index=True)
endpoint_times = endpoint_times.groupby("node_id", sort=False)["first_seen"].min()
node_first_seen = pd.concat([node_first_seen, endpoint_times]).groupby(level=0).min()

print(f"Unique process observations: {len(node_rows):,}")
print(f"Unique parent/child edges:   {len(edge_table):,}")
display(edge_table.head())

Unique process observations: 104,282
Unique parent/child edges:   117,929


,parent,child,first_seen,last_seen,n_events,event_files,relation_types
0,7dmf69pk-0c587426-00001960-00000000-1da94c8547...,7dmf69pk-0c587426-00001a44-00000000-1da94c854a...,2024-04-22 15:18:29.505752600+00:00,2024-04-22 22:14:08.735031400+00:00,1667,"(h1,)","(parent_guid->process_guid, process_guid->chil..."
1,7dmf69pk-0c587426-00000388-00000000-1da94c81d6...,7dmf69pk-0c587426-00000728-00000000-1da94c81df...,2024-04-22 15:17:08.488392100+00:00,2024-04-22 22:11:39.300558800+00:00,339,"(h1,)","(parent_guid->process_guid, process_guid->chil..."
2,7dmf69pk-0c587426-000001f8-00000000-1da94c81da...,7dmf69pk-0c587426-00001a3c-00000000-1da94c854a...,2024-04-22 15:18:29.505752600+00:00,2024-04-22 15:18:30.853475900+00:00,19,"(h1,)","(parent_guid->process_guid, process_guid->chil..."
3,7dmf69pk-0c587426-00000c70-00000000-1da94c8509...,7dmf69pk-0c587426-000019e8-00000000-1da94c8549...,2024-04-22 15:18:29.435309900+00:00,2024-04-22 15:18:30.953734600+00:00,7,"(h1,)","(parent_guid->process_guid, process_guid->chil..."
4,7dmf69pk-0c587426-00001928-00000000-1da94c8547...,7dmf69pk-0c587426-000019b8-00000000-1da94c8548...,2024-04-22 15:18:29.383033900+00:00,2024-04-22 22:14:08.797531200+00:00,63,"(h1,)","(parent_guid->process_guid, process_guid->chil..."


## Prepare malicious/non-malicious and explanation labels

In [6]:
def unique_strings(values: pd.Series) -> tuple[str, ...]:
    cleaned = [str(value).strip().lower() for value in values.dropna() if str(value).strip()]
    return tuple(dict.fromkeys(cleaned))


def join_explanations(values: pd.Series) -> str:
    cleaned = [str(value).strip() for value in values.dropna() if str(value).strip()]
    return " | ".join(dict.fromkeys(cleaned)) or "None"


alerts = alert_labels.copy()
alerts["_guid"] = normalize_guid(alerts[ALERT_GUID])
alerts["_class"] = alerts[ALERT_CLASS].astype("string").str.strip().str.lower()
alerts["_explanation"] = alerts[ALERT_EXPLANATION]
alerts = alerts.dropna(subset=["_guid"])

alert_by_guid = alerts.groupby("_guid", sort=False).agg(
    alert_labels=("_class", unique_strings),
    explanation=("_explanation", join_explanations),
)
alert_by_guid["is_malicious"] = alert_by_guid["alert_labels"].map(
    lambda labels: any(label in MALICIOUS_LABELS for label in labels)
)
alert_by_guid["in_alert_labels"] = True

print(f"Alert-labelled process GUIDs: {len(alert_by_guid):,}")
display(alert_by_guid.head())

Alert-labelled process GUIDs: 7,745


,alert_labels,explanation,is_malicious,in_alert_labels
_guid,,,,
7dmf69pk-0c483285-000007a3-00000000-1da94bd81fe8b65,['benign'],Execution - Unix Shell,False,True
7dmf69pk-0c483285-000007a3-00000000-1da94bd81fee508,['benign'],Execution - Unix Shell,False,True
7dmf69pk-0c47c1fe-00000684-00000000-1da8f824f66ce8b,['benign'],Credential Access - LLMNR/NBT-NS Poisoning - L...,False,True
7dmf69pk-0c483285-00000e32-00000000-1da94c5e3ff9cb7,['benign'],Execution - Unix Shell,False,True
7dmf69pk-0c483285-00000e32-00000000-1da94c5e40011ae,['benign'],Execution - Unix Shell,False,True


## Build and label the full directed process graph

In [7]:
def clean_value(value):
    if isinstance(value, pd.Timestamp):
        return value.to_pydatetime()
    if pd.api.types.is_scalar(value) and pd.isna(value):
        return None
    return value


G = nx.DiGraph(
    dataset=DATASET_NAME,
    common_event_columns=tuple(common_event_columns),
)

# ZZX

# Build the complete graph first. Cleanup is performed once, after all node
# timestamps and edge attributes have been attached.
for edge in edge_table.itertuples(index=False):
    G.add_edge(
        str(edge.parent),
        str(edge.child),
        first_seen=clean_value(edge.first_seen),
        last_seen=clean_value(edge.last_seen),
        n_events=int(edge.n_events),
        event_files=edge.event_files,
        relation_types=edge.relation_types,
    )

# Include processes that have no retained parent/child edge.
G.add_nodes_from(str(node_id) for node_id in node_rows.index)

common_defaults = {column: None for column in common_event_columns}

for node_id in G.nodes:
    attributes = common_defaults.copy()
    attributes[PROCESS_GUID] = node_id

    if node_id in node_rows.index:
        attributes.update({
            column: clean_value(node_rows.at[node_id, column])
            for column in node_value_columns
        })

    attributes["first_seen"] = clean_value(node_first_seen.get(node_id, pd.NaT))
    attributes["event_count"] = int(node_counts.get(node_id, 0))
    attributes["event_files"] = node_event_files.get(node_id, tuple())

    if node_id in alert_by_guid.index:
        alert = alert_by_guid.loc[node_id]
        attributes["in_alert_labels"] = True
        attributes["alert_labels"] = alert["alert_labels"]
        attributes["is_malicious"] = bool(alert["is_malicious"])
        attributes["explanation"] = alert["explanation"]
    else:
        attributes["in_alert_labels"] = False
        attributes["alert_labels"] = tuple()
        attributes["is_malicious"] = False
        attributes["explanation"] = "None"

    attributes["malicious_label"] = (
        "malicious" if attributes["is_malicious"] else "non-malicious"
    )
    G.nodes[node_id].update(attributes)


def timestamp_ns(value):
    """Return a comparable UTC nanosecond timestamp, or None when missing."""
    timestamp = pd.to_datetime(value, errors="coerce", utc=True)
    return None if pd.isna(timestamp) else int(timestamp.value)


nodes_before_cleanup = G.number_of_nodes()
edges_before_cleanup = G.number_of_edges()

# 1. Remove every vertex without a valid timestamp.
node_time = {
    node: timestamp_ns(attributes.get("first_seen"))
    for node, attributes in G.nodes(data=True)
}
nodes_without_timestamps = {
    node for node, timestamp in node_time.items()
    if timestamp is None
}
G.remove_nodes_from(nodes_without_timestamps)
node_time = {
    node: timestamp
    for node, timestamp in node_time.items()
    if node in G
}

# 2. Retain only edges that move strictly forward in node time. This also
# removes self-loops and guarantees that the resulting directed graph is a DAG.
non_forward_edges = [
    (parent, child)
    for parent, child in G.edges
    if node_time[parent] >= node_time[child]
]
G.remove_edges_from(non_forward_edges)
assert nx.is_directed_acyclic_graph(G)

# 3. Within each weak component, keep its oldest root and retain only nodes
# reachable from that root. Branches belonging only to younger roots are pruned.
nodes_pruned_with_younger_roots = set()
root_prune_rows = []

for component_id, component_nodes in enumerate(
    list(nx.weakly_connected_components(G))
):
    component_nodes = set(component_nodes)
    component = G.subgraph(component_nodes)
    roots = [
        node for node, in_degree in component.in_degree()
        if in_degree == 0
    ]
    assert roots  # Every finite DAG component has at least one root.

    kept_root = min(
        roots,
        key=lambda node: (node_time[node], str(node)),
    )
    retained_nodes = nx.descendants(component, kept_root) | {kept_root}
    dropped_nodes = component_nodes - retained_nodes
    nodes_pruned_with_younger_roots.update(dropped_nodes)

    if len(roots) > 1:
        root_prune_rows.append({
            "component_id": component_id,
            "roots_before": len(roots),
            "kept_root": kept_root,
            "kept_root_time": G.nodes[kept_root]["first_seen"],
            "dropped_roots": tuple(
                root for root in roots if root != kept_root
            ),
            "dropped_nodes": len(dropped_nodes),
        })

G.remove_nodes_from(nodes_pruned_with_younger_roots)
node_time = {
    node: timestamp
    for node, timestamp in node_time.items()
    if node in G
}
root_prune_report = pd.DataFrame(root_prune_rows)

assert all(
    sum(
        in_degree == 0
        for _, in_degree in G.subgraph(component_nodes).in_degree()
    ) == 1
    for component_nodes in nx.weakly_connected_components(G)
)

# 4. A rooted DAG fails to be a tree exactly when a non-root has multiple
# incoming edges. Keep its earliest observed incoming edge, thereby deleting
# redundant cycle edges in reverse-timestamp order without creating new roots.
def incoming_edge_priority(edge):
    parent, child, attributes = edge
    edge_time = timestamp_ns(attributes.get("first_seen"))
    return (
        edge_time is None,
        0 if edge_time is None else edge_time,
        node_time[parent],
        -int(attributes.get("n_events", 0)),
        str(parent),
        str(child),
    )


redundant_edges_to_remove = []
redundant_edge_rows = []

for component_nodes in nx.weakly_connected_components(G):
    component = G.subgraph(component_nodes)
    roots = [
        node for node, in_degree in component.in_degree()
        if in_degree == 0
    ]
    assert len(roots) == 1
    root = roots[0]

    for child in component.nodes:
        if child == root:
            continue

        incoming_edges = list(component.in_edges(child, data=True))
        assert incoming_edges

        kept_edge = min(incoming_edges, key=incoming_edge_priority)
        kept_parent = kept_edge[0]

        for parent, _, attributes in incoming_edges:
            if parent == kept_parent:
                continue
            redundant_edges_to_remove.append((parent, child))
            redundant_edge_rows.append({
                "parent": parent,
                "child": child,
                "first_seen": attributes.get("first_seen"),
                "kept_parent": kept_parent,
            })

G.remove_edges_from(redundant_edges_to_remove)
removed_redundant_edges = pd.DataFrame(redundant_edge_rows)

# Final guarantees: all edges move forward in time and every weak component is
# a directed rooted tree (an arborescence).
assert not G.is_multigraph()
assert nx.is_directed_acyclic_graph(G)
assert all(
    node_time[parent] < node_time[child]
    for parent, child in G.edges
)
assert all(
    nx.is_arborescence(G.subgraph(component_nodes))
    for component_nodes in nx.weakly_connected_components(G)
)

print(f"Removed vertices without timestamps: {len(nodes_without_timestamps):,}")
print(f"Removed non-forward edges:           {len(non_forward_edges):,}")
print(f"Pruned younger-root vertices:        {len(nodes_pruned_with_younger_roots):,}")
print(f"Removed redundant cycle edges:       {len(redundant_edges_to_remove):,}")
print(
    f"Cleanup: {nodes_before_cleanup:,} nodes / {edges_before_cleanup:,} edges"
    f" -> {G.number_of_nodes():,} nodes / {G.number_of_edges():,} edges"
)

if not root_prune_report.empty:
    display(root_prune_report.head(20))
if not removed_redundant_edges.empty:
    display(removed_redundant_edges.head(20))

# ZZY

print(f"Full graph nodes: {G.number_of_nodes():,}")
print(f"Full graph edges: {G.number_of_edges():,}")
print(f"Process trees/components: {nx.number_weakly_connected_components(G):,}")

/tmp/ipykernel_1078322/1552990546.py:3: UserWarning: Discarding nonzero nanoseconds in conversion.
  return value.to_pydatetime()
/tmp/ipykernel_1078322/1552990546.py:3: UserWarning: Discarding nonzero nanoseconds in conversion.
  return value.to_pydatetime()
/tmp/ipykernel_1078322/1552990546.py:3: UserWarning: Discarding nonzero nanoseconds in conversion.
  return value.to_pydatetime()


Removed vertices without timestamps: 0
Removed non-forward edges:           4,672
Pruned younger-root vertices:        0
Removed redundant cycle edges:       857
Cleanup: 117,335 nodes / 117,929 edges -> 117,335 nodes / 112,400 edges


,parent,child,first_seen,kept_parent
0,7dmf69pk-0c587426-00002428-00000334-1da94dbf71...,7dmf69pk-0c587426-0000173c-00000334-1da94fdc5a...,2024-04-22 21:41:10.728692+00:00,7dmf69pk-0c587426-00002428-00000000-1da94dbf71...
1,7dmf69pk-0c587426-000005d0-00000000-1da94c9448...,7dmf69pk-0c587426-00001c5c-00000000-1da94c944c...,2024-04-22 15:40:13.615041+00:00,7dmf69pk-0c587426-000005d0-00000318-1da94c9448...
2,7dmf69pk-0c587426-00002428-00000000-1da94dbf71...,7dmf69pk-0c587426-00002fc8-00000000-1da94dbf76...,2024-04-22 17:44:03.375095+00:00,7dmf69pk-0c587426-00002428-00000334-1da94dbf71...
3,7dmf69pk-0c587426-00001f5c-00000000-1da9592dbf...,7dmf69pk-0c587426-00000638-00000000-1da9592dc4...,2024-04-23 16:28:19.821340+00:00,7dmf69pk-0c587426-00001f5c-000002b4-1da9592dbf...
4,7dmf69pk-0c587426-00001814-00000000-1da967ddda...,7dmf69pk-0c587426-00001b20-00000000-1da967dddf...,2024-04-24 19:45:56.679476+00:00,7dmf69pk-0c587426-00001814-00000398-1da967ddda...
5,7dmf69pk-0c587426-00000fb8-00000000-1da965bb79...,7dmf69pk-0c587426-0000042c-00000000-1da965bb7e...,2024-04-24 15:56:13.460214+00:00,7dmf69pk-0c587426-00000fb8-000003f0-1da965bb79...
6,7dmf69pk-0c587426-00002db4-00000000-1da9670c39...,7dmf69pk-0c587426-000032ac-00000000-1da9670c3d...,2024-04-24 18:05:16.414769+00:00,7dmf69pk-0c587426-00002db4-0000009c-1da9670c39...
7,7dmf69pk-0c587426-000011e4-00000000-1da9680d49...,7dmf69pk-0c587426-00002a2c-00000000-1da9680d4d...,2024-04-24 20:51:55.498788+00:00,7dmf69pk-0c587426-000011e4-0000033c-1da9680d49...
8,7dmf69pk-0c587426-00000fb8-00000000-1da965bb79...,7dmf69pk-0c587426-00002a10-000003f0-1da965f794...,2024-04-24 15:52:57.168652+00:00,7dmf69pk-0c587426-00000fb8-000003f0-1da965bb79...
9,7dmf69pk-0c587426-00002d18-00000000-1da97fe20d...,7dmf69pk-0c587426-00001870-00000000-1da97fe213...,2024-04-26 17:36:29.649362+00:00,7dmf69pk-0c587426-00002d18-0000035c-1da97fe20d...


Full graph nodes: 117,335
Full graph edges: 112,400
Process trees/components: 4,935


## Save one pickle per process tree/component

A component root is an in-degree-zero process. When a component has more than one such process, the earliest one is used for the filename and graph metadata. If a component has no in-degree-zero process, the earliest process is used. Repeated node-count/date combinations receive `_2`, `_3`, and so on to prevent overwriting.

In [8]:
def timestamp_key(value) -> int:
    if value is None:
        return 2**63 - 1
    timestamp = pd.Timestamp(value)
    return 2**63 - 1 if pd.isna(timestamp) else int(timestamp.value)


def choose_root(graph: nx.DiGraph) -> str:
    roots = [node for node, degree in graph.in_degree() if degree == 0]
    candidates = roots or list(graph.nodes)
    return min(
        candidates,
        key=lambda node: (timestamp_key(graph.nodes[node].get("first_seen")), str(node)),
    )


component_records = []
for nodes in nx.weakly_connected_components(G):
    graph = G.subgraph(nodes).copy()

    roots = [node for node in graph.nodes if graph.in_degree(node) == 0]

    if len(roots) == 1:
        root = choose_root(graph)
        root_time = graph.nodes[root].get("first_seen")
        component_records.append({
            "graph": graph,
            "root": root,
            "root_time": root_time,
            "root_time_key": timestamp_key(root_time),
            "n_nodes": graph.number_of_nodes(),
            "n_edges": graph.number_of_edges(),
        })

component_records.sort(key=lambda row: (row["root_time_key"], row["root"], row["n_nodes"]))
name_counts = {}
graph_index_rows = []

for component_number, record in enumerate(component_records, start=1):
    root_timestamp = pd.Timestamp(record["root_time"])
    date_text = "unknown" if pd.isna(root_timestamp) else root_timestamp.strftime("%d_%m_%Y")
    stem = f"{DATASET_NAME}_graph_n{record['n_nodes']}_d{date_text}"

    name_counts[stem] = name_counts.get(stem, 0) + 1
    occurrence = name_counts[stem]
    filename = f"{stem}.pkl" if occurrence == 1 else f"{stem}_{occurrence}.pkl"
    output_path = GRAPH_DIR / filename

    graph = record["graph"]
    graph.graph.update({
        "dataset": DATASET_NAME,
        "component_number": component_number,
        "root": record["root"],
        "root_time": clean_value(record["root_time"]),
        "n_nodes": record["n_nodes"],
        "n_edges": record["n_edges"],
        "common_event_columns": tuple(common_event_columns),
    })

    with output_path.open("wb") as handle:
        pickle.dump(graph, handle, protocol=pickle.HIGHEST_PROTOCOL)

    graph_index_rows.append({
        "file": filename,
        "root": record["root"],
        "root_time": record["root_time"],
        "n_nodes": record["n_nodes"],
        "n_edges": record["n_edges"],
        "n_malicious_nodes": sum(
            bool(data.get("is_malicious")) for _, data in graph.nodes(data=True)
        ),
    })

graph_index = pd.DataFrame(graph_index_rows)
graph_index_path = GRAPH_DIR / f"{DATASET_NAME}_graph_index.csv"
graph_index.to_csv(graph_index_path, index=False)

print(f"Saved {len(graph_index):,} graph pickle(s) in {GRAPH_DIR}")
print(f"Saved graph index to {graph_index_path}")
display(graph_index.head(20))

Saved 4,935 graph pickle(s) in ../../data/CARBANAK/graphs
Saved graph index to ../../data/CARBANAK/graphs/CARBANAK_graph_index.csv


,file,root,root_time,n_nodes,n_edges,n_malicious_nodes
0,CARBANAK_graph_n64_d18_04_2024.pkl,7dmf69pk-0c51f386-000003d8-00000000-1da904ebe1...,2024-04-18 19:08:52.961710+00:00,64,63,0
1,CARBANAK_graph_n1_d18_04_2024.pkl,7dmf69pk-0c51f386-0000290c-00000000-1da91c3da5...,2024-04-18 19:08:52.961710+00:00,1,0,0
2,CARBANAK_graph_n132_d18_04_2024.pkl,7dmf69pk-0c51f386-00000348-00000000-1da904ebdd...,2024-04-18 19:15:34.947972+00:00,132,131,0
3,CARBANAK_graph_n20_d18_04_2024.pkl,7dmf69pk-0c51f386-0000141c-00000000-1da904ebfc...,2024-04-18 19:15:34.947972+00:00,20,19,0
4,CARBANAK_graph_n1_d18_04_2024_2.pkl,7dmf69pk-0c51f386-00002088-00000000-1da91c4ca0...,2024-04-18 19:15:34.947972+00:00,1,0,0
5,CARBANAK_graph_n10_d18_04_2024.pkl,7dmf69pk-0c51f386-00002d7c-00000000-1da91c5016...,2024-04-18 19:17:14.275611+00:00,10,9,0
6,CARBANAK_graph_n1_d18_04_2024_3.pkl,7dmf69pk-0c51f386-00002f14-00000000-1da91c500d...,2024-04-18 19:17:14.275611+00:00,1,0,0
7,CARBANAK_graph_n108_d18_04_2024.pkl,7dmf69pk-0c51f386-000009c8-00000000-1da91c3c84...,2024-04-18 19:17:14.786930+00:00,108,107,0
8,CARBANAK_graph_n1_d18_04_2024_4.pkl,7dmf69pk-0c51f386-000026f0-00000000-1da91c3d4d...,2024-04-18 19:17:14.786930+00:00,1,0,0
9,CARBANAK_graph_n1_d18_04_2024_5.pkl,7dmf69pk-0c51f386-000010a0-00000000-1da91c3c81...,2024-04-18 19:17:14.941916+00:00,1,0,0


## Load one saved graph

In [9]:
# Example:
# graph_path = GRAPH_DIR / graph_index.iloc[0]["file"]
# with graph_path.open("rb") as handle:
#     process_tree = pickle.load(handle)
#
# print(process_tree.graph)
# first_node = next(iter(process_tree.nodes))
# print(first_node, process_tree.nodes[first_node])